# Vollautomatisches RAG- & Update-Skript
## Hier ist das vollautomatische Skript. Es überwacht einen lokalen Ordner namens quell_dokumente.Beim Start analysiert das Skript alle enthaltenen Textdateien. Es fügt vollautomatisch nur die Dateien hinzu, die noch nicht in der Datenbank existieren, um zeitaufwendige doppelte Embeddings zu vermeiden.
## Vollautomatisches RAG- & Update-Skript

 mein_rag_projekt/
 │
 ├── hauptskript.py                      # Dein Python-Code
 │
 ├── dokumente_mietrecht_paragraph/                    # Hier liegen deine Originaltexte
 │   ├── mietrecht_kuendigung_ganz_para.txt
 │   ├── mietrecht_mietzahlung_ganz_para.txt
 │   
 │
 └── chroma_mietrecht_paragraph_qwen/         # Der von Chroma generierte Datenbank-Ordner
    ├── chroma.sqlite3                  # SQLite-Datenbank (enthält Metadaten & Relationen)
    └── b3a5f7-9c.../                   # UUID-Ordner (enthält die tatsächlichen Vektor-Indizes)
        ├── data_level0.bin
        ├── header.bin
        └── link_lists.bin



In [2]:
import os
import shutil
from langchain_community.document_loaders import TextLoader
from langchain_community.llms import Ollama
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.vectorstores import Chroma 
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document

# =====================================================================
# CONFIGURATION
# =====================================================================
PERSIST_DIRECTORY = "chroma_mietrecht_paragraph_qwen"  # Neuer sauberer Ordnername!
DOCUMENTS_DIR = "dokumente_mietrecht_paragraph"

os.makedirs(DOCUMENTS_DIR, exist_ok=True)

# Wir nutzen Nomic für das große Kontextfenster (8k)
embeddings = OllamaEmbeddings(model="nomic-embed-text") 
llm = Ollama(model="qwen3.5:9b")

# HARD RESET (Optional): Falls du die DB komplett frisch aufbauen willst,
# aktiviere die nächste Zeile (Raute entfernen), um alte Fehler zu löschen:
# if os.path.exists(PERSIST_DIRECTORY): shutil.rmtree(PERSIST_DIRECTORY)

# =====================================================================
# SYSTEMATISCHES EINLESEN
# =====================================================================
if 'vector_store' not in globals():
    print("🤖 Initialisiere saubere Chroma-Datenbank...")
    vector_store = Chroma(
        persist_directory=PERSIST_DIRECTORY,
        embedding_function=embeddings
    )
else:
    print("🔄 Nutze bestehende Verbindung.")

# Bereits verarbeitete Dateien ermitteln
existierende_daten = vector_store.get()
bereits_eingelesen = set()
if existierende_daten and "metadatas" in existierende_daten:
    for meta in existierende_daten["metadatas"]:
        if meta and "source" in meta:
            bereits_eingelesen.add(os.path.basename(meta["source"]))

alle_dateien = [f for f in os.listdir(DOCUMENTS_DIR) if f.endswith(".txt")]
neue_dateien = [f for f in alle_dateien if f not in bereits_eingelesen]

if neue_dateien:
    print(f"✨ Neue Dokumente gefunden: {neue_dateien}")
    for datei_name in neue_dateien:
        datei_pfad = os.path.join(DOCUMENTS_DIR, datei_name)
        print(f"📄 Indiziere: {datei_name}...")
        
        try:
            with open(datei_pfad, "r", encoding="utf-8") as f:
                volltext = f.read()
            
            # Präziser Split nach unseren Paragraphen-Markern
            rohe_paragraphen = volltext.split("=== PARAGRAPH_START:")
            chunks = []
            
            for raw_chunk in rohe_paragraphen:
                text_inhalt = raw_chunk.strip()
                if not text_inhalt:
                    continue
                if "=== PARAGRAPH_END ===" in text_inhalt:
                    text_inhalt = text_inhalt.split("=== PARAGRAPH_END ===")[0].strip()
                
                chunks.append(Document(page_content=text_inhalt, metadata={"source": datei_pfad}))
            
            if chunks:
                vector_store.add_documents(documents=chunks)
                print(f"✅ {datei_name} erfolgreich mit {len(chunks)} Chunks eingelesen.")
        except Exception as e:
            print(f"❌ Fehler bei {datei_name}: {e}")
else:
    print("ℹ️ Vektordatenbank ist auf dem neuesten Stand.")

# =====================================================================
# OPTIMIERTER RETRIEVER & PROMPT
# =====================================================================
# WICHTIG: k=5, damit das RAG-System MEHRERE Paragraphen gleichzeitig an Qwen übergibt!
retriever = vector_store.as_retriever(search_kwargs={"k": 5})

system_prompt = (
    "Du bist ein präziser Rechtsassistent für deutsches Mietrecht.\n"
    "Dir werden mehrere Paragraphen als Kontext bereitgestellt. Sie enthalten jeweils "
    "[ORIGINALTEXT], [ERKLÄRUNG] und [TYPISCHE_NUTZERFRAGEN].\n\n"
    "Prüfe alle übergebenen Abschnitte sorgfältig. Beantworte die Frage des Nutzers "
    "wahrheitsgemäß und nenne den exakten Paragraphen aus dem [ORIGINALTEXT].\n"
    "Wenn die Antwort im bereitgestellten Kontext nicht zu finden ist, sage klipp und klar: "
    "'Ich weiß es nicht, da der passende Gesetzestext im Kontext fehlt.'\n\n"
    "Kontext:\n{context}"
)

prompt_template = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

question_answer_chain = create_stuff_documents_chain(llm, prompt_template)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

# =====================================================================
# ABFRAGE
# =====================================================================
such_query = "Ich wohne seit 6 Jahren in meiner Wohnung. Welche Kündigungsfrist gilt für meinen Vermieter?"
print(f"\nFrage: {such_query}\nBerechne Antwort...\n")

antworte_objekt = rag_chain.invoke({"input": such_query})
print("=== ANTWORT VON QWEN ===")
print(antworte_objekt["answer"])


C:\Users\wug2si\AppData\Local\Temp\ipykernel_28492\2325039710.py:21: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  embeddings = OllamaEmbeddings(model="nomic-embed-text")
C:\Users\wug2si\AppData\Local\Temp\ipykernel_28492\2325039710.py:22: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm = Ollama(model="qwen3.5:9b")
C:\Users\wug2si\AppData\Local\Temp\ipykernel_28492\2325039710.py:33: LangChainDeprecationWarning: The class `Chrom

🤖 Initialisiere saubere Chroma-Datenbank...
ℹ️ Vektordatenbank ist auf dem neuesten Stand.

Frage: Ich wohne seit 6 Jahren in meiner Wohnung. Welche Kündigungsfrist gilt für meinen Vermieter?
Berechne Antwort...

=== ANTWORT VON QWEN ===
Basierend auf den bereitgestellten Gesetzestexten gilt für Ihre Situation Folgendes:

Nach **§ 573c BGB Abs. 1** ist die Kündigungsfrist für den Vermieter bei einer Wohndauer von mehr als fünf Jahren verlängert. Der Text erklärt dazu: „Wohnt der Mieter bereits 5 Jahre dort, beträgt die Frist für den Vermieter 6 Monate. Ab 8 Jahren Wohndauer beträgt sie 9 Monate."

Da Sie seit **6 Jahren** in der Wohnung wohnen, liegt Ihre Mietdauer zwischen dem Fünften und dem Achten Jahr.

**Ergebnis:**
Für Ihren Vermieter gilt eine verlängerte Kündigungsfrist von **6 Monaten**.
(Die Grundfrist beträgt 3 Monate, sie verlängert sich um jeweils 3 Monate nach 5 und 8 Jahren).

Zusätzlich beachten Sie die Kündigungsfrist des Vermieters:
*   Die Kündigung muss **spätestens

In [3]:
such_query = "Ich wohne seit 3 Jahren in meiner Wohnung. Welche Kündigungsfrist gilt für meinen Vermieter?"
print(f"\nFrage: {such_query}\nBerechne Antwort...\n")

antworte_objekt = rag_chain.invoke({"input": such_query})
print("=== ANTWORT VON QWEN ===")
print(antworte_objekt["answer"])


Frage: Ich wohne seit 3 Jahren in meiner Wohnung. Welche Kündigungsfrist gilt für meinen Vermieter?
Berechne Antwort...

=== ANTWORT VON QWEN ===

